# Asthma Attack Prediction

*Simple data understanding and preparation for the AAMOS-00 dataset.*

In [1]:
# Import the libraries we will use
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Load the datasets

In [2]:
# Load the four AAMOS-00 datasets
daily = pd.read_csv("data/anonym_aamos00_dailyquestionnaire.csv")
weekly = pd.read_csv("data/anonym_aamos00_weeklyquestionnaire.csv")
peakflow = pd.read_csv("data/anonym_aamos00_peakflow.csv")
patient = pd.read_csv("data/anonym_aamos00_patient_info.csv")

print("Daily:", daily.shape)
print("Weekly:", weekly.shape)
print("Peak flow:", peakflow.shape)
print("Patient:", patient.shape)

Daily: (1583, 8)
Weekly: (324, 12)
Peak flow: (1516, 6)
Patient: (22, 28)


In [3]:
# Check the columns so we know what each dataset contains
print("DAILY", daily.columns.tolist())
print("\nWEEKLY", weekly.columns.tolist())
print("\nPEAK FLOW", peakflow.columns.tolist())
print("\nPATIENT", patient.columns.tolist())

DAILY ['user_key', 'date', 'daily_night_symp', 'daily_day_symp', 'daily_limit_activity', 'daily_prev_inhaler', 'daily_relief_inhaler', 'daily_triggers']

WEEKLY ['user_key', 'date', 'weekly_night_symp', 'weekly_day_symp', 'weekly_limit_activity', 'weekly_short_breath', 'weekly_wheeze', 'weekly_relief_inhaler', 'weekly_doc', 'weekly_hospital', 'weekly_er', 'weekly_oral']

PEAK FLOW ['user_key', 'date', 'hour', 'pef_max', 'pefs', 'morning']

PATIENT ['user_key', 'sex', 'age_range', 'bmi_range', 'age_diagnosed_range', 'max_pef_expected', 'smoker', 'pack_years', 'race', 'severity', 'pef_best', 'n_inhalers', 'region', 'nation', 'phase2_start', 'daily_start_date', 'weekly_start_date', 'miband_start_date', 'position_start_date', 'pef_start_date', 'inhaler_start_date', 'daily_end_date', 'weekly_end_date', 'miband_end_date', 'position_end_date', 'pef_end_date', 'inhaler_end_date', 'phase2_end']


In [4]:
# Quickly look at the first few rows of each dataset
display(daily.head())
display(weekly.head())
display(peakflow.head())
display(patient.head())

,user_key,date,daily_night_symp,daily_day_symp,daily_limit_activity,daily_prev_inhaler,daily_relief_inhaler,daily_triggers
0,113,0,True,True,False,2,5,"5,9"
1,113,1,False,False,False,2,3,"5,9"
2,113,2,False,True,False,2,5,"5,9"
3,113,3,False,False,False,2,1,"5,12"
4,113,4,False,True,False,2,3,"5,9"


,user_key,date,weekly_night_symp,weekly_day_symp,weekly_limit_activity,weekly_short_breath,weekly_wheeze,weekly_relief_inhaler,weekly_doc,weekly_hospital,weekly_er,weekly_oral
0,113,1,2.0,3.0,1.0,2,2,3,-4,0,0,2.0
1,113,7,2.0,3.0,2.0,2,2,2,-5,0,0,2.0
2,113,14,5.0,5.0,5.0,4,4,5,"-1,-3",0,-2,3.0
3,113,22,4.0,5.0,4.0,4,2,4,-3,0,0,3.0
4,113,28,3.0,3.0,3.0,3,2,3,0,0,0,3.0


,user_key,date,hour,pef_max,pefs,morning
0,113,1,7,469,469|99|144,True
1,113,1,15,310,310|174|175,False
2,113,3,13,290,249|290|207,True
3,113,40,16,557,71|557|525,False
4,190,6,7,333,333|307|150,True


,user_key,sex,age_range,bmi_range,age_diagnosed_range,max_pef_expected,smoker,pack_years,race,severity,...,position_start_date,pef_start_date,inhaler_start_date,daily_end_date,weekly_end_date,miband_end_date,position_end_date,pef_end_date,inhaler_end_date,phase2_end
0,113,female,30-39yo,Obesity,7-11yo,485,Never (<100 cigarettes in lifetime),0.00,White,Severe,...,1,1.0,0.0,183,183,183.0,183,40.0,177.0,183
1,190,male,50+yo,Pre-obesity,19+yo,575,Previous,18.75,White,Severe,...,1,6.0,1.0,183,182,148.0,183,183.0,19.0,183
2,217,female,30-39yo,Obesity,19+yo,495,Previous,6.30,White,Moderate,...,0,0.0,1.0,1,0,1.0,1,0.0,1.0,1
3,278,female,18-29yo,Pre-obesity,12-18yo,495,Previous,1.20,White,Moderate,...,0,NaN,-20.0,15,15,15.0,13,NaN,106.0,106
4,294,female,50+yo,Obesity,0-6yo,455,Never (<100 cigarettes in lifetime),0.00,White,Moderate,...,0,0.0,-5.0,183,182,183.0,183,183.0,184.0,184


## 2. Understand the weekly questionnaire

In [5]:
# Load the data dictionary once so we can verify variable meanings
data_dict = pd.read_excel(
    "data/aamos00_data_dictionary.xlsx",
    sheet_name="data_dictionary"
)

weekly_vars = [
    "weekly_night_symp",
    "weekly_day_symp",
    "weekly_limit_activity",
    "weekly_short_breath",
    "weekly_wheeze",
    "weekly_oral",
    "weekly_relief_inhaler",
    "weekly_doc",
    "weekly_hospital",
    "weekly_er"
]

# Show the definitions and coding for the weekly variables
weekly_dict = data_dict[data_dict["Variable Name"].isin(weekly_vars)]
display(weekly_dict[["Variable Name", "Variable Label", "Answer Details", "Data type", "Format"]])

,Variable Name,Variable Label,Answer Details,Data type,Format
149,weekly_night_symp,How many days over the last week have you had ...,Integer from 0 to 7,Numeric,N
150,weekly_day_symp,How many days over the last week have you had ...,Integer from 0 to 7,Numeric,N
151,weekly_limit_activity,How many days over the last week has your asth...,Integer from 0 to 7,Numeric,N
152,weekly_short_breath,Which of the following statement best describe...,Possible values:\n1 = I did not experience any...,Numeric,N
153,weekly_wheeze,Which of the following statement best describe...,Possible values:\n1 = I did not wheeze due to ...,Numeric,N
154,weekly_relief_inhaler,Which of the following statement best describe...,Possible values:\n1 = I had not used my relief...,Numeric,N
155,weekly_doc,Did you visit a general practice (GP) or asthm...,Multiple choice. Possible values:\n0 = No\n-1 ...,Numeric,N
156,weekly_hospital,Were you hospitalised due to your asthma in th...,Multiple choice. Possible values:\n0 = No\n-1 ...,Numeric,N
157,weekly_er,Did you visit an emergency room (ER) for your ...,Multiple choice. Possible values:\n0 = No\n-1 ...,Numeric,N
158,weekly_oral,Did you use systemic corticosteroids (Predniso...,"Possible values: 1 = No, 2 = Yes, but no more ...",Numeric,N


In [6]:
# Check the target variable before creating it
print(weekly["weekly_oral"].value_counts(dropna=False).sort_index())

weekly_oral
1.0    241
2.0     39
3.0     43
NaN      1
Name: count, dtype: int64


In [7]:
# Look at the weekly symptom ranges and missing values
symptom_cols = [
    "weekly_night_symp",
    "weekly_day_symp",
    "weekly_limit_activity",
    "weekly_short_breath",
    "weekly_wheeze"
]

print(weekly[symptom_cols].describe())

for col in symptom_cols:
    print(f"\n--- {col} ---")
    print(weekly[col].value_counts(dropna=False).sort_index())

       weekly_night_symp  weekly_day_symp  weekly_limit_activity  \
count         148.000000       291.000000             209.000000   
mean            2.568919         5.364261               3.913876   
std             1.737417         2.115125               2.221453   
min             1.000000         1.000000               1.000000   
25%             1.000000         4.000000               2.000000   
50%             2.000000         7.000000               3.000000   
75%             3.000000         7.000000               6.000000   
max             7.000000         7.000000               7.000000   

       weekly_short_breath  weekly_wheeze  
count           324.000000     324.000000  
mean              2.398148       1.851852  
std               0.930460       0.867167  
min               1.000000       1.000000  
25%               2.000000       1.000000  
50%               2.000000       2.000000  
75%               3.000000       2.000000  
max               5.000000       5.

In [8]:
# Check the unusual value against the documented 0–7 range
display(weekly[weekly["weekly_night_symp"] == 1.2])

,user_key,date,weekly_night_symp,weekly_day_symp,weekly_limit_activity,weekly_short_breath,weekly_wheeze,weekly_relief_inhaler,weekly_doc,weekly_hospital,weekly_er,weekly_oral
16,113,113,1.2,3.0,2.0,2,1,2,-3,0,0,2.0


In [9]:
# Check missing values before preprocessing
print(weekly.isna().sum())

user_key                   0
date                       0
weekly_night_symp        176
weekly_day_symp           33
weekly_limit_activity    115
weekly_short_breath        0
weekly_wheeze              0
weekly_relief_inhaler      0
weekly_doc                 1
weekly_hospital            0
weekly_er                  0
weekly_oral                1
dtype: int64


## 3. Create the classification target

In [10]:
# 1 = corticosteroid use was more than usual, 0 = otherwise
# Keep the missing target as missing instead of turning it into 0
weekly_model = weekly.copy()
weekly_model["target"] = weekly_model["weekly_oral"].map(
    lambda x: 1 if x == 3 else 0 if pd.notna(x) else np.nan
)

print(weekly_model["target"].value_counts(dropna=False))
print("\nPercentages:")
print(weekly_model["target"].value_counts(normalize=True, dropna=True) * 100)

target
0.0    280
1.0     43
NaN      1
Name: count, dtype: int64

Percentages:
target
0.0    86.687307
1.0    13.312693
Name: proportion, dtype: float64


## 4. Check participant information

In [11]:
# Check missing values in participant-level information
print(patient.isna().sum())

user_key               0
sex                    0
age_range              0
bmi_range              0
age_diagnosed_range    1
max_pef_expected       0
smoker                 0
pack_years             0
race                   0
severity               0
pef_best               1
n_inhalers             1
region                 0
nation                 0
phase2_start           0
daily_start_date       0
weekly_start_date      0
miband_start_date      2
position_start_date    0
pef_start_date         6
inhaler_start_date     5
daily_end_date         0
weekly_end_date        0
miband_end_date        2
position_end_date      0
pef_end_date           6
inhaler_end_date       5
phase2_end             0
dtype: int64


In [12]:
# Inspect the main participant characteristics
patient_cols = [
    "sex", "age_range", "bmi_range", "age_diagnosed_range",
    "smoker", "severity", "n_inhalers"
]

for col in patient_cols:
    print(f"\n--- {col} ---")
    print(patient[col].value_counts(dropna=False))


--- sex ---
sex
female    17
male       5
Name: count, dtype: int64

--- age_range ---
age_range
30-39yo    9
40-49yo    6
50+yo      4
18-29yo    3
Name: count, dtype: int64

--- bmi_range ---
bmi_range
Obesity        9
Normal         7
Pre-obesity    6
Name: count, dtype: int64

--- age_diagnosed_range ---
age_diagnosed_range
0-6yo      9
7-11yo     5
19+yo      5
12-18yo    2
NaN        1
Name: count, dtype: int64

--- smoker ---
smoker
Never (<100 cigarettes in lifetime)    17
Previous                                5
Name: count, dtype: int64

--- severity ---
severity
Moderate       12
Severe          7
Mild            2
Very severe     1
Name: count, dtype: int64

--- n_inhalers ---
n_inhalers
1.0     12
2.0      8
10.0     1
NaN      1
Name: count, dtype: int64


## 5. Check peak-flow data

In [14]:
# Get a quick summary of the peak-flow measurements
print(peakflow.describe())
display(peakflow.head())

          user_key         date         hour      pef_max
count  1516.000000  1516.000000  1516.000000  1516.000000
mean    480.914908    69.709103    13.246702   397.932058
std     245.093118    50.787506     6.313678    93.872577
min     113.000000    -2.000000     0.000000   120.000000
25%     294.000000    26.000000     7.000000   318.000000
50%     447.000000    59.000000    11.000000   399.000000
75%     702.000000   109.000000    20.000000   461.000000
max     939.000000   183.000000    23.000000   639.000000


,user_key,date,hour,pef_max,pefs,morning
0,113,1,7,469,469|99|144,True
1,113,1,15,310,310|174|175,False
2,113,3,13,290,249|290|207,True
3,113,40,16,557,71|557|525,False
4,190,6,7,333,333|307|150,True


In [16]:
# Check peak flow by participant
peakflow.groupby("user_key")["pef_max"].agg(["count", "mean", "min", "max"])

,count,mean,min,max
user_key,,,,
113,4,406.500000,290,557
190,230,327.386957,143,639
217,1,544.000000,544,544
294,291,409.518900,267,508
328,22,504.000000,480,532
343,194,379.969072,120,508
447,49,295.469388,207,359
473,140,478.557143,342,559
514,129,431.767442,302,523


In [17]:
# Check expected peak flow
patient[["user_key", "max_pef_expected"]]

,user_key,max_pef_expected
0,113,485
1,190,575
2,217,495
3,278,495
4,294,455
5,328,505
6,343,475
7,398,500
8,447,465
9,454,455


In [18]:
print(weekly[["user_key", "date"]].head(10))
print()
print(peakflow[["user_key", "date"]].head(10))

   user_key  date
0       113     1
1       113     7
2       113    14
3       113    22
4       113    28
5       113    35
6       113    42
7       113    49
8       113    57
9       113    63

   user_key  date
0       113     1
1       113     1
2       113     3
3       113    40
4       190     6
5       190     7
6       190     8
7       190     9
8       190    10
9       190    11


In [19]:
# Check how many peak-flow readings fall within the previous 7 days
weekly_dates = weekly[["user_key", "date"]].copy()

counts = []

for _, row in weekly_dates.iterrows():
    user = row["user_key"]
    day = row["date"]

    n = peakflow[
        (peakflow["user_key"] == user) &
        (peakflow["date"] < day) &
        (peakflow["date"] >= day - 7)
    ].shape[0]

    counts.append(n)

weekly_dates["peakflow_7day_count"] = counts

print(weekly_dates["peakflow_7day_count"].describe())
print("\nWeeks with no peak-flow readings:")
print((weekly_dates["peakflow_7day_count"] == 0).sum())

count    324.000000
mean       4.453704
std        4.624041
min        0.000000
25%        0.000000
50%        3.000000
75%        8.000000
max       14.000000
Name: peakflow_7day_count, dtype: float64

Weeks with no peak-flow readings:
126


In [20]:
# Start with the weekly data
model_data = weekly_model.copy()

# Add participant information
model_data = model_data.merge(
    patient,
    on="user_key",
    how="left"
)

print(model_data.shape)
print(model_data.head())

(324, 40)
   user_key  date  weekly_night_symp  weekly_day_symp  weekly_limit_activity  \
0       113     1                2.0              3.0                    1.0   
1       113     7                2.0              3.0                    2.0   
2       113    14                5.0              5.0                    5.0   
3       113    22                4.0              5.0                    4.0   
4       113    28                3.0              3.0                    3.0   

   weekly_short_breath  weekly_wheeze  weekly_relief_inhaler weekly_doc  \
0                    2              2                      3         -4   
1                    2              2                      2         -5   
2                    4              4                      5      -1,-3   
3                    4              2                      4         -3   
4                    3              2                      3          0   

   weekly_hospital  ...  position_start_date  pef_start_da

In [21]:
# Keep the variables we actually need
features = [
    "weekly_night_symp",
    "weekly_day_symp",
    "weekly_limit_activity",
    "weekly_short_breath",
    "weekly_wheeze",
    "weekly_relief_inhaler",
    "weekly_doc",
    "weekly_hospital",
    "weekly_er",
    "sex",
    "age_range",
    "bmi_range",
    "smoker",
    "severity",
    "target"
]

model_data = model_data[features]

print(model_data.shape)
print(model_data.head())

(324, 15)
   weekly_night_symp  weekly_day_symp  weekly_limit_activity  \
0                2.0              3.0                    1.0   
1                2.0              3.0                    2.0   
2                5.0              5.0                    5.0   
3                4.0              5.0                    4.0   
4                3.0              3.0                    3.0   

   weekly_short_breath  weekly_wheeze  weekly_relief_inhaler weekly_doc  \
0                    2              2                      3         -4   
1                    2              2                      2         -5   
2                    4              4                      5      -1,-3   
3                    4              2                      4         -3   
4                    3              2                      3          0   

   weekly_hospital  weekly_er     sex age_range bmi_range  \
0                0          0  female   30-39yo   Obesity   
1                0          0  f

In [22]:
# Check how the healthcare visit variables are stored
for col in ["weekly_doc", "weekly_hospital", "weekly_er"]:
    print("\n", col)
    print(model_data[col].value_counts(dropna=False).head(15))


 weekly_doc
weekly_doc
0        280
-3         8
-4         7
-2         7
-1         7
-5         5
-3,-6      2
-7         2
-1,-3      1
-4,-2      1
NaN        1
-6         1
-3,-1      1
-2,-3      1
Name: count, dtype: int64

 weekly_hospital
weekly_hospital
 0    318
-2      2
-1      1
-7      1
-5      1
-4      1
Name: count, dtype: int64

 weekly_er
weekly_er
 0    317
-2      2
-1      2
-7      1
-5      1
-4      1
Name: count, dtype: int64


In [23]:
# Remove variables that need complicated event-date handling
model_data = model_data.drop(
    columns=["weekly_doc", "weekly_hospital", "weekly_er"]
)

print(model_data.shape)
print(model_data.columns.tolist())

(324, 12)
['weekly_night_symp', 'weekly_day_symp', 'weekly_limit_activity', 'weekly_short_breath', 'weekly_wheeze', 'weekly_relief_inhaler', 'sex', 'age_range', 'bmi_range', 'smoker', 'severity', 'target']


In [24]:
# Check missing values in our final feature set
print(model_data.isnull().sum())

weekly_night_symp        176
weekly_day_symp           33
weekly_limit_activity    115
weekly_short_breath        0
weekly_wheeze              0
weekly_relief_inhaler      0
sex                        0
age_range                  0
bmi_range                  0
smoker                     0
severity                   0
target                     1
dtype: int64


In [25]:
# See how many rows have all symptom values available
symptom_cols = [
    "weekly_night_symp",
    "weekly_day_symp",
    "weekly_limit_activity"
]

complete_rows = model_data[symptom_cols].notnull().all(axis=1)

print("Complete symptom rows:", complete_rows.sum())
print("Rows with missing symptoms:", (~complete_rows).sum())

Complete symptom rows: 134
Rows with missing symptoms: 190


In [26]:
# Remove the one observation where the target is unknown
model_data = model_data.dropna(subset=["target"])

print(model_data.shape)
print(model_data["target"].value_counts())

(323, 12)
target
0.0    280
1.0     43
Name: count, dtype: int64


In [27]:
# Check the median of the missing symptom variables
symptom_cols = [
    "weekly_night_symp",
    "weekly_day_symp",
    "weekly_limit_activity"
]

print(model_data[symptom_cols].median())

weekly_night_symp        2.0
weekly_day_symp          7.0
weekly_limit_activity    3.0
dtype: float64


In [28]:
# Fill missing symptom values with their median
for col in symptom_cols:
    model_data[col] = model_data[col].fillna(model_data[col].median())

# Check that the missing values are gone
print(model_data[symptom_cols].isnull().sum())

weekly_night_symp        0
weekly_day_symp          0
weekly_limit_activity    0
dtype: int64


In [29]:
# Final missing-value check
print(model_data.isnull().sum())

weekly_night_symp        0
weekly_day_symp          0
weekly_limit_activity    0
weekly_short_breath      0
weekly_wheeze            0
weekly_relief_inhaler    0
sex                      0
age_range                0
bmi_range                0
smoker                   0
severity                 0
target                   0
dtype: int64


In [30]:
# Check the categories in each categorical feature
categorical_cols = [
    "sex",
    "age_range",
    "bmi_range",
    "smoker",
    "severity"
]

for col in categorical_cols:
    print("\n", col)
    print(model_data[col].value_counts())


 sex
sex
female    235
male       88
Name: count, dtype: int64

 age_range
age_range
30-39yo    121
40-49yo    114
50+yo       69
18-29yo     19
Name: count, dtype: int64

 bmi_range
bmi_range
Normal         120
Obesity        104
Pre-obesity     99
Name: count, dtype: int64

 smoker
smoker
Never (<100 cigarettes in lifetime)    254
Previous                                69
Name: count, dtype: int64

 severity
severity
Moderate       166
Severe         118
Mild            38
Very severe      1
Name: count, dtype: int64


In [31]:
# Convert categorical variables into numerical columns
model_data = pd.get_dummies(
    model_data,
    columns=categorical_cols,
    drop_first=True
)

print(model_data.shape)
print(model_data.columns.tolist())

(323, 17)
['weekly_night_symp', 'weekly_day_symp', 'weekly_limit_activity', 'weekly_short_breath', 'weekly_wheeze', 'weekly_relief_inhaler', 'target', 'sex_male', 'age_range_30-39yo', 'age_range_40-49yo', 'age_range_50+yo', 'bmi_range_Obesity', 'bmi_range_Pre-obesity', 'smoker_Previous', 'severity_Moderate', 'severity_Severe', 'severity_Very severe']


The ML Part Begins here

In [32]:
# Separate input features and target
X = model_data.drop(columns=["target"])
y = model_data["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (323, 16)
y shape: (323,)


In [33]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("\nTraining target:")
print(y_train.value_counts())
print("\nTesting target:")
print(y_test.value_counts())

Training set: (258, 16)
Testing set: (65, 16)

Training target:
target
0.0    224
1.0     34
Name: count, dtype: int64

Testing target:
target
0.0    56
1.0     9
Name: count, dtype: int64


In [34]:
from sklearn.naive_bayes import GaussianNB

# Create and train the Naive Bayes model
model = GaussianNB()

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [35]:
# Predict the target for the test data
y_pred = model.predict(X_test)

print(y_pred)

[1. 1. 1. 1. 0. 1. 1. 1. 1. 0. 0. 0. 1. 0. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0.
 0. 0. 1. 0. 1. 1. 1. 1. 0. 1. 0. 1. 1. 0. 1. 1. 1. 1. 1. 1. 0. 0. 1. 0.
 1. 1. 1. 0. 1. 0. 0. 0. 0. 1. 1. 0. 1. 1. 0. 1. 1.]


In [36]:
from sklearn.metrics import accuracy_score, confusion_matrix

# Compare predictions with the actual test values
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nConfusion Matrix:")
print(cm)

Accuracy: 0.49230769230769234

Confusion Matrix:
[[24 32]
 [ 1  8]]


In [37]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.96      0.43      0.59        56
         1.0       0.20      0.89      0.33         9

    accuracy                           0.49        65
   macro avg       0.58      0.66      0.46        65
weighted avg       0.85      0.49      0.56        65



In [38]:
# Majority-class baseline
baseline_accuracy = (y_test == 0).mean()

print("Baseline accuracy:", baseline_accuracy)

Baseline accuracy: 0.8615384615384616


In [39]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Try a few smoothing values
values = [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3]

for value in values:
    model = GaussianNB(var_smoothing=value)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(value, "->", accuracy)

1e-09 -> 0.49230769230769234
1e-08 -> 0.5692307692307692
1e-07 -> 0.6461538461538462
1e-06 -> 0.676923076923077
1e-05 -> 0.7076923076923077
0.0001 -> 0.7384615384615385
0.001 -> 0.7846153846153846


In [40]:
#big imporvement, we went from 0.49 to 0.75 accuracy just by tuning var_smoothing.
#but we're still below the 0.86 majority baseline.

# Test larger smoothing values
values = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1]

for value in values:
    model = GaussianNB(var_smoothing=value)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(value, "->", accuracy)

0.001 -> 0.7846153846153846
0.002 -> 0.7846153846153846
0.005 -> 0.7846153846153846
0.01 -> 0.7692307692307693
0.02 -> 0.7846153846153846
0.05 -> 0.7846153846153846
0.1 -> 0.8153846153846154


In [43]:
#we are still below the majority baseline, but we are getting closer. 0.1 smoothing gives us 0.81 accuracy.

#more importantly, this experiment tells us that,
#the default Naive Bayes assumptions are not fitting this dataset well

# Test a wider range of smoothing values
values = [0.1, 0.2, 0.5, 1, 2, 5, 10]

for value in values:
    model = GaussianNB(var_smoothing=value)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(value, "->", accuracy)

0.1 -> 0.8153846153846154
0.2 -> 0.8
0.5 -> 0.8
1 -> 0.8615384615384616
2 -> 0.8615384615384616
5 -> 0.8615384615384616
10 -> 0.8615384615384616


at var_smoothing = 1 and above, the model reaches 86.15%, exactly the majority-class baseline


| Approach            |   Accuracy |
| ------------------- | ---------: |
| Default Naive Bayes |     49.23% |
| Best so far (`0.1`) | **81.54%** |
| Majority baseline   | **86.15%** |
| Naive Bayes at `≥1` |     86.15% |